In [1]:
import pandas as pd
import re
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_json("../data/processed/kpi_dataset.json")

# choose your text column
texts = df["caption_text"].fillna("").astype(str).tolist()

c:\univ\Data mining\Project\venv-bert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
print(os.getcwd())

c:\univ\Data mining\Project\notebooks


In [3]:
def clean_caption(text):
    text = str(text).lower()

    # remove URLs, mentions
    text = re.sub(r"http\S+|www\S+|@\w+", " ", text)

    # remove phone numbers
    text = re.sub(r"\b\d{5,}\b", " ", text)

    # normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # remove tatweel and diacritics
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # remove extra symbols
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_caption"] = df["caption_text"].fillna("").apply(clean_caption)
texts = df["clean_caption"].tolist()

In [4]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import os

# 1) Local cache directory inside project
LOCAL_MODEL_DIR = Path("./.local_models/sentence_transformers/all-MiniLM-L6-v2")
LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

# 2) Load from local if exists, otherwise download once then save locally
if LOCAL_MODEL_DIR.exists() and any(LOCAL_MODEL_DIR.iterdir()):
    model = SentenceTransformer(str(LOCAL_MODEL_DIR), local_files_only=True)
else:
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    model.save(str(LOCAL_MODEL_DIR))  # saves full model locally for reuse

print(f"Using model from: {LOCAL_MODEL_DIR.resolve()}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11444.97it/s]

Using model from: C:\univ\Data mining\Project\notebooks\.local_models\sentence_transformers\all-MiniLM-L6-v2


In [6]:
arabic_stop_words = [
    "في", "من", "على", "الى", "إلى", "عن", "مع", "هذا", "هذه", "ذلك",
    "هو", "هي", "هم", "ما", "لا", "نعم", "او", "أو", "كل", "تم",
    "الله", "ان", "إن", "أن", "كان", "كانت", "بعد", "قبل",

    # social/business noise
    "متوفر", "متوفره", "متوفرة", "عرض", "عروض", "خصم", "السعر",
    "اسعار", "للطلب", "للتواصل", "تواصل", "رسائل", "الصفحه",
    "الصفحة", "واتساب", "رقم", "اتصال", "التوصيل", "دليفري",
    "اليوم", "جديد", "جديدنا", "تابعونا", "زورونا",

    # location noise, only remove if location is not your goal
    # "رام", "رام الله", "جنين", "نابلس", "البيره", "البيرة"
]

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    stop_words=arabic_stop_words
)

topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    min_topic_size=15,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(texts)

2026-05-13 22:19:53,692 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 23/23 [00:07<00:00,  3.13it/s]
2026-05-13 22:20:01,059 - BERTopic - Embedding - Completed ✓
2026-05-13 22:20:01,059 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-13 22:20:11,866 - BERTopic - Dimensionality - Completed ✓
2026-05-13 22:20:11,867 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-13 22:20:11,931 - BERTopic - Cluster - Completed ✓
2026-05-13 22:20:11,935 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-13 22:20:11,986 - BERTopic - Representation - Completed ✓


In [7]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,184,-1_بجانب_العيد_العنوان_لكل,"[بجانب, العيد, العنوان, لكل, 24, شارع, مقابل, ...",[بنطلون خامه مميزه توصيل خلال 24 ساعه العنوان ...
1,0,89,0_شيكل_فانيلا_اليوم_وكل_يوم_فانيلا_الافضل,"[شيكل, فانيلا_اليوم_وكل_يوم, فانيلا, الافضل, ا...",[طقم العيد بناتي قماش مرتب بسعر 40 شيكل، توصيل...
2,1,65,1_مقابل_علي_اللي_شارع,"[مقابل, علي, اللي, شارع, غزه, النصر, جنين, كاز...",[الخيارات عنا بتحيرك بين برجر وبوفلك شاورما، و...
3,2,59,2_the_كنتاكي_دجاج كنتاكي_like,"[the, كنتاكي, دجاج كنتاكي, like, of, دجاج, and...",[ما يعدل المود غير ال good food دجاج كنتاكي لذ...
4,3,42,3_شيقل_متوفر_توصيل_لجميع_مناطق_الضفه_القدس_وال...,"[شيقل, متوفر_توصيل_لجميع_مناطق_الضفه_القدس_وال...",[اساسيات الجامعه الكتب اضافه من يورومودا وهيك ...
5,4,41,4_الموقع_dco بجانب_البيره الطلوع_الطلوع قرب,"[الموقع, dco بجانب, البيره الطلوع, الطلوع قرب,...",[كل ما تكبر في مطعم نور تجربه رايقه ومناخ لا ي...
6,5,40,5_شارع_مزاج_وصل_لجميع المناطق,"[شارع, مزاج, وصل, لجميع المناطق, العنوان, المن...",[لا تفوتوا فرصه الاستمتاع باطباق مزاج الشرقي ف...
7,6,39,6_المحافظات اقل_المحافظات_اقل_محدوده,"[المحافظات اقل, المحافظات, اقل, محدوده, الكميه...",[وصلت احلي بضاعه للعيد لحقي حالك الكميه جدا مح...
8,7,26,7_الساعه الطابق_جنين برج_برج_برج الساعه,"[الساعه الطابق, جنين برج, برج, برج الساعه, الا...",[شنط مميزه شغل يدوي مميز جنين برج الساعه الطاب...
9,8,24,8_the_gaza_to_in,"[the, gaza, to, in, of, and, this, on, are, from]",[one stitch at a time one hand to the next 100...


In [8]:
topic_model.get_topic(0)

[('شيكل', np.float64(0.0702779282269316)),
 ('فانيلا_اليوم_وكل_يوم', np.float64(0.06975082594138478)),
 ('فانيلا', np.float64(0.06214700146752159)),
 ('الافضل', np.float64(0.05719260553772259)),
 ('الوان', np.float64(0.05595701352652223)),
 ('شو', np.float64(0.05362373983850315)),
 ('طقم', np.float64(0.04650055062758984)),
 ('فانيلا فانيلا_اليوم_وكل_يوم', np.float64(0.0459568905603956)),
 ('القدس شقفاط', np.float64(0.0459568905603956)),
 ('الوان اقل', np.float64(0.0459568905603956))]

In [9]:
topic_model.visualize_topics()

In [10]:
topic_model.visualize_barchart(top_n_topics=10)

In [39]:
df_clean = df[df["caption_text"].fillna("").astype(str).str.len() > 5].copy()
df_clean["topic_id"] = topics

topic_info = topic_model.get_topic_info()
df_clean = df_clean.merge(
    topic_info[["Topic", "Name"]],
    left_on="topic_id",
    right_on="Topic",
    how="left"
)

df_clean.to_csv("dataset_with_topics.csv", index=False)

In [40]:
topic_info = topic_model.get_topic_info()

for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue

    keywords = topic_model.get_topic(topic_id)
    docs = topic_model.get_representative_docs(topic_id)

    print("\nTopic:", topic_id)
    print("Keywords:", keywords[:10])
    print("Examples:", docs[:3])


Topic: 0
Keywords: [('the', np.float64(0.06641563255265041)), ('gaza', np.float64(0.06034477058792767)), ('of', np.float64(0.0578593460122206)), ('in', np.float64(0.05681257820162266)), ('to', np.float64(0.05359140098930889)), ('0566000650', np.float64(0.04284935737091275)), ('0599420500', np.float64(0.04284935737091275)), ('الشرقي', np.float64(0.039994803967967)), ('and', np.float64(0.03884851969232496)), ('الثورة', np.float64(0.038564421633821476))]
Examples: ['المذاق الفاخر يتميز به في جميع الأصناف الشرقية و الغربية في مزاج الشرقي. للحجز والاستفسار للتواصل: 0599420500، 0566000650، 082825003 العنوان: غزة، تقاطع شارع النصر مع شارع الثورة - بناية بدري وهبة.', 'استمتع بتجربة متميزة مع مزاج الشرقي ونكهاته الشهية والأصناف الشرقية الأصيلة والمتنوعة. للحجز والاستفسار للتواصل: 0599420500، 0566000650، 082825003 العنوان: غزة، تقاطع شارع النصر مع شارع الثورة - بناية بدري وهبة.', 'استمتعوا بلذة الأصناف الغربية من مزاج الشرقي. للحجز والاستفسار التواصل: 0599420500، 0566000650، 082825003 العنوان: 